# watsonx.ai SaaS × GitHub Integration — Proof of Concept

**Region:** `us-south` (Dallas)  
**Platform:** IBM Cloud (watsonx.ai SaaS)  
**Git provider:** GitHub

---

## What this notebook demonstrates

| Step | What happens |
|------|--------------|
| 1 | Authenticate against IBM Cloud IAM and obtain a bearer token |
| 2 | Call the watsonx.ai API to confirm project connectivity |
| 3 | Load the sample churn CSV dataset |
| 4 | Train a RandomForest classifier and print metrics |
| 5 | Write `model_summary.json` — the artefact that gets pushed to GitHub |
| 6 | Use the GitHub REST API to commit `model_summary.json` directly to the repo |

Steps 1–5 prove the **wx.ai side**. Step 6 proves the **Git side** and closes the round-trip.

> **Credentials:** Copy `.env.example` → `.env`, fill in values, and upload `.env` as a data asset named `env` in this project before running.

## 0. Load credentials from the `.env` data asset

In [ ]:
import os
from dotenv import load_dotenv

# Inside a wx.ai SaaS runtime, data assets live at /project_data/data_asset/
env_path = "/project_data/data_asset/env"
if os.path.exists(env_path):
    load_dotenv(env_path)
    print("✅  Loaded credentials from data asset")
else:
    # Local dev fallback
    load_dotenv(".env")
    print("⚠️   Running locally — loaded from .env file")

IBM_CLOUD_API_KEY = os.getenv("IBM_CLOUD_API_KEY")
WX_PROJECT_ID     = os.getenv("WX_PROJECT_ID")
IAM_URL           = os.getenv("IAM_URL", "https://iam.cloud.ibm.com/identity/token")
WX_URL            = os.getenv("WX_URL",  "https://us-south.ml.cloud.ibm.com")
GITHUB_TOKEN      = os.getenv("GITHUB_TOKEN")
GITHUB_REPO       = os.getenv("GITHUB_REPO")   # e.g. "my-org/wxai-git-poc"

assert IBM_CLOUD_API_KEY, "IBM_CLOUD_API_KEY is not set"
assert WX_PROJECT_ID,     "WX_PROJECT_ID is not set"
assert GITHUB_TOKEN,      "GITHUB_TOKEN is not set"
assert GITHUB_REPO,       "GITHUB_REPO is not set"

print(f"Project ID : {WX_PROJECT_ID}")
print(f"GitHub repo: {GITHUB_REPO}")

## 1. Authenticate — IBM Cloud IAM bearer token

In [ ]:
import requests

iam_resp = requests.post(
    IAM_URL,
    headers={"Content-Type": "application/x-www-form-urlencoded"},
    data={
        "grant_type":    "urn:ibm:params:oauth:grant-type:apikey",
        "apikey":        IBM_CLOUD_API_KEY,
    },
)
iam_resp.raise_for_status()
IAM_TOKEN = iam_resp.json()["access_token"]

WX_HEADERS = {
    "Authorization": f"Bearer {IAM_TOKEN}",
    "Content-Type":  "application/json",
}
print("✅  IAM token obtained")

## 2. Verify watsonx.ai project is reachable

In [ ]:
proj_resp = requests.get(
    f"{WX_URL}/v2/projects/{WX_PROJECT_ID}",
    headers=WX_HEADERS,
)
proj_resp.raise_for_status()
proj = proj_resp.json()

print(f"✅  Connected to project: {proj['entity']['name']}")
print(f"   Created : {proj['metadata'].get('created_at', 'n/a')}")
print(f"   Owner   : {proj['entity'].get('creator_iam_id', 'n/a')}")

## 3. Load the churn dataset

In [ ]:
import pandas as pd

ASSET_PATH = "/project_data/data_asset/wxo-generated-churn-data.csv"
LOCAL_PATH  = "../wxo-generated-churn-data.csv"

if os.path.exists(ASSET_PATH):
    df = pd.read_csv(ASSET_PATH)
    print(f"✅  Loaded from data asset ({len(df)} rows)")
elif os.path.exists(LOCAL_PATH):
    df = pd.read_csv(LOCAL_PATH)
    print(f"✅  Loaded from local path ({len(df)} rows)")
else:
    raise FileNotFoundError(
        "Upload 'wxo-generated-churn-data.csv' as a data asset in the wx.ai project."
    )

df.head()

## 4. Train a churn classifier

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report

df_enc = df.copy()
for col in ["gender", "region", "product_category"]:
    df_enc[col] = LabelEncoder().fit_transform(df_enc[col])
df_enc["churned"] = (df_enc["churned"] == "Yes").astype(int)

FEATURES = [
    "age", "gender", "income", "account_balance",
    "region", "product_category",
    "usage_minutes_per_month", "complaints_last_year",
]
TARGET = "churned"

X_train, X_test, y_train, y_test = train_test_split(
    df_enc[FEATURES], df_enc[TARGET],
    test_size=0.25, random_state=42, stratify=df_enc[TARGET]
)

clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)

acc    = accuracy_score(y_test, clf.predict(X_test))
report = classification_report(
    y_test, clf.predict(X_test), target_names=["No Churn", "Churn"]
)

print(f"Accuracy : {acc:.4f}")
print(report)

## 5. Write model summary artefact

In [ ]:
import json
from datetime import datetime, timezone

summary = {
    "model":        "RandomForestClassifier",
    "n_estimators": 100,
    "features":     FEATURES,
    "accuracy":     round(acc, 4),
    "dataset_rows": len(df),
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "wx_project_id": WX_PROJECT_ID,
    "git_note":     "Auto-committed by wxai_git_poc.ipynb via GitHub API",
}

SUMMARY_FILE = "model_summary.json"
with open(SUMMARY_FILE, "w") as f:
    json.dump(summary, f, indent=2)

print(f"✅  Wrote {SUMMARY_FILE}")
print(json.dumps(summary, indent=2))

## 6. Push `model_summary.json` to GitHub (API commit)

This cell uses the **GitHub Contents API** to create or update the file directly in the repo — no `git` CLI needed inside the runtime.

In [ ]:
import base64

GH_API  = "https://api.github.com"
GH_PATH = f"{GH_API}/repos/{GITHUB_REPO}/contents/{SUMMARY_FILE}"
GH_HEADERS = {
    "Authorization": f"Bearer {GITHUB_TOKEN}",
    "Accept":        "application/vnd.github+json",
    "X-GitHub-Api-Version": "2022-11-28",
}

# Check if file already exists (need its SHA to update)
existing = requests.get(GH_PATH, headers=GH_HEADERS)
sha = existing.json().get("sha") if existing.status_code == 200 else None

with open(SUMMARY_FILE, "rb") as f:
    content_b64 = base64.b64encode(f.read()).decode()

payload = {
    "message": f"chore: update model_summary.json — accuracy={summary['accuracy']}",
    "content": content_b64,
    "branch":  "main",
}
if sha:
    payload["sha"] = sha   # required when updating an existing file

push_resp = requests.put(GH_PATH, headers=GH_HEADERS, json=payload)
push_resp.raise_for_status()

html_url = push_resp.json()["content"]["html_url"]
action   = "Updated" if sha else "Created"
print(f"✅  {action} file in GitHub: {html_url}")

## 7. Summary

| Step | Result |
|------|--------|
| IAM auth | Bearer token from `iam.cloud.ibm.com` |
| wx.ai project API | Project name confirmed via REST |
| Data load | Churn CSV loaded from project data asset |
| ML training | RandomForest — accuracy printed above |
| Git commit | `model_summary.json` pushed to `main` branch via GitHub API |

**The PoC is complete.** You can now connect the same GitHub repo to this watsonx.ai project under *Project → Manage → Git integration* to also pull/push the notebook itself.